# Low-risk versus high-risk CRC XGBoost model

This notebook reproduces the second layer of the analysis. Stages 0–2 are coded as low risk (`0`) and stages 3–4 as high risk (`1`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb

from matplotlib.patches import Polygon
from sklearn.base import clone
from sklearn.metrics import (
    RocCurveDisplay,
    auc,
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.validation import check_is_fitted

## File locations

In [ ]:
project_root = Path("..")
data_dir = project_root / "data" / "processed"
figure_dir = project_root / "results" / "figures"
shap_output_dir = project_root / "results" / "shap" / "risk_model"

figure_dir.mkdir(parents=True, exist_ok=True)
shap_output_dir.mkdir(parents=True, exist_ok=True)

## Load and prepare the CRC training data

In [ ]:
train_data = pd.read_table(
    data_dir / "train_exp_selected_mirs.txt",
    sep=" "
)

crc_train_data = train_data.loc[
    train_data.loc[:, "stage"] != "Healthy",
    :
].copy()

crc_train_expression = crc_train_data.iloc[:, :24].copy()

crc_train_stages = crc_train_data.loc[:, "stage"].copy()
crc_train_stages = np.array(list(map(int, crc_train_stages)))

print("CRC training data shape:", crc_train_expression.shape)

In [ ]:
risk_groups = []

for stage in crc_train_stages:
    if stage in (0, 1, 2):
        risk_groups.append("LR")
    elif stage in (3, 4):
        risk_groups.append("HR")

risk_train_labels = pd.Series(risk_groups)
print(risk_train_labels.value_counts())

risk_train_labels = [
    0 if risk_group == "LR" else 1
    for risk_group in risk_train_labels
]

print("High-risk training samples:", sum(risk_train_labels))

## Rank candidate miRNAs and select the feature subset

In [ ]:
feature_ranking_model = xgb.XGBClassifier(
    objective="binary:logistic",
    random_state=1
)

feature_ranking_model.fit(
    crc_train_expression,
    risk_train_labels
)

feature_importance = feature_ranking_model.feature_importances_

feature_importance_table = pd.DataFrame({
    "Feature": crc_train_expression.columns,
    "Importance": feature_importance
})

feature_importance_table = feature_importance_table.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

feature_importance_table

In [ ]:
subset_model = xgb.XGBClassifier(
    objective="binary:logistic",
    random_state=1
)

subset_f1_scores = []

for number_of_features in range(1, 25):
    selected_features = feature_importance_table["Feature"][
        :number_of_features
    ]

    subset_train_expression = crc_train_expression.loc[
        :,
        selected_features
    ]

    cv_scores = cross_val_score(
        subset_model,
        subset_train_expression,
        risk_train_labels,
        cv=5,
        scoring="f1_macro"
    )

    subset_f1_scores.append(cv_scores.mean())

feature_importance_table["f1"] = subset_f1_scores
feature_importance_table

In [ ]:
fig, importance_axis = plt.subplots(figsize=(10, 6))
plt.rcParams["font.family"] = "Times"

sns.barplot(
    x="Feature",
    y="Importance",
    data=feature_importance_table,
    ax=importance_axis,
    palette=sns.color_palette("Blues", n_colors=24),
    order=feature_importance_table["Feature"],
    hue=feature_importance_table["Importance"],
    legend=False
)

importance_axis.set_ylabel("miRNAs Importance", fontsize=17)
importance_axis.set_xlabel("")
importance_axis.set_xticklabels(
    importance_axis.get_xticklabels(),
    rotation=35,
    ha="right"
)
importance_axis.tick_params(
    axis="both",
    which="major",
    labelsize=14
)

f1_axis = importance_axis.twinx()

sns.lineplot(
    x="Feature",
    y="f1",
    data=feature_importance_table,
    ax=f1_axis,
    color="red",
    marker="o",
    sort=False
)

f1_axis.set_ylabel(
    "F1 Score",
    fontsize=17,
    rotation=270,
    labelpad=18
)
f1_axis.tick_params(axis="both", which="major", labelsize=14)
f1_axis.axvline(x=11, color="red", linestyle="--")

plt.tight_layout()
plt.savefig(
    figure_dir / "risk_feature_importance_and_F1.pdf",
    bbox_inches="tight"
)
plt.show()

## Load the CRC hold-out test data

In [ ]:
test_data = pd.read_table(
    data_dir / "test_exp_selected_mirs.txt",
    sep=" "
)

crc_test_data = test_data.loc[
    test_data.loc[:, "stage"] != "Healthy",
    :
].copy()

crc_test_expression = crc_test_data.iloc[:, :24].copy()

crc_test_stages = crc_test_data.loc[:, "stage"].copy()
crc_test_stages = np.array(list(map(int, crc_test_stages)))

risk_test_groups = []

for stage in crc_test_stages:
    if stage in (0, 1, 2):
        risk_test_groups.append("LR")
    elif stage in (3, 4):
        risk_test_groups.append("HR")

risk_test_labels = [
    0 if risk_group == "LR" else 1
    for risk_group in risk_test_groups
]

print("CRC test data shape:", crc_test_expression.shape)
print("High-risk test samples:", sum(risk_test_labels))

## Train and evaluate the final 12-miRNA model

In [ ]:
final_features = feature_importance_table["Feature"][:12]

risk_train_expression = crc_train_expression.loc[:, final_features]
risk_test_expression = crc_test_expression.loc[:, final_features]

print("Final risk-panel miRNAs:")
print(list(final_features))

In [ ]:
class_weight_by_label = {
    0: 0.9114252061248527,
    1: 9.082451253481894
}

risk_model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=7
)

training_sample_weights = [
    class_weight_by_label[label]
    for label in risk_train_labels
]

risk_model.fit(
    risk_train_expression,
    risk_train_labels,
    sample_weight=training_sample_weights
)

test_predictions = risk_model.predict(risk_test_expression)

cv_scores = cross_val_score(
    risk_model,
    risk_train_expression,
    risk_train_labels,
    cv=5,
    scoring="f1_macro"
)

print("Mean five-fold macro F1:", cv_scores.mean())
print(
    "Training and test accuracy:",
    risk_model.score(risk_train_expression, risk_train_labels),
    risk_model.score(risk_test_expression, risk_test_labels)
)
print(classification_report(risk_test_labels, test_predictions))
print(
    "Test macro F1:",
    f1_score(risk_test_labels, test_predictions, average="macro")
)

## Bootstrap confidence intervals

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score

def bootstrap_ci(y_true, y_prob, threshold=0.5,
                 n_bootstraps=2000, random_state=42):

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    rng = np.random.default_rng(random_state)

    auc_values = []
    f1_macro_values = []

    n = len(y_true)

    for _ in range(n_bootstraps):

        indices = rng.integers(0, n, size=n)

        y_boot = y_true[indices]
        prob_boot = y_prob[indices]

        # Skip samples containing only one class
        if len(np.unique(y_boot)) < 2:
            continue

        pred_boot = (prob_boot >= threshold).astype(int)

        auc_values.append(
            roc_auc_score(y_boot, prob_boot)
        )

        f1_macro_values.append(
            f1_score(y_boot, pred_boot, average="macro")
        )

    auc_ci = np.percentile(auc_values, [2.5, 97.5])
    f1_ci = np.percentile(f1_macro_values, [2.5, 97.5])

    auc = roc_auc_score(y_true, y_prob)

    y_pred = (y_prob >= threshold).astype(int)
    f1_macro = f1_score(y_true, y_pred, average="macro")

    print(
        f"AUROC: {auc:.2f} "
        f"(95% CI: {auc_ci[0]:.2f}–{auc_ci[1]:.2f})"
    )

    print(
        f"Macro F1: {f1_macro:.2f} "
        f"(95% CI: {f1_ci[0]:.2f}–{f1_ci[1]:.2f})"
    )

In [ ]:
test_probabilities = risk_model.predict_proba(
    risk_test_expression
)[:, 1]

bootstrap_ci(
    y_true=risk_test_labels,
    y_prob=test_probabilities,
    threshold=0.5
)

## Receiver operating characteristic curve

In [ ]:
roc_cv = StratifiedKFold(n_splits=5)

fold_tprs = []
fold_aucs = []
mean_fpr = np.linspace(0, 1, 100)

for fold, (train_index, validation_index) in enumerate(
    roc_cv.split(risk_train_expression, risk_train_labels)
):
    fold_train_labels = [
        risk_train_labels[i]
        for i in train_index
    ]

    fold_sample_weights = [
        class_weight_by_label[label]
        for label in fold_train_labels
    ]

    risk_model.fit(
        risk_train_expression.iloc[train_index, :],
        fold_train_labels,
        sample_weight=fold_sample_weights
    )

    validation_probabilities = risk_model.predict_proba(
        risk_train_expression.iloc[validation_index, :]
    )[:, 1]

    roc_display = RocCurveDisplay.from_predictions(
        [risk_train_labels[i] for i in validation_index],
        validation_probabilities
    )

    interpolated_tpr = np.interp(
        mean_fpr,
        roc_display.fpr,
        roc_display.tpr
    )
    interpolated_tpr[0] = 0.0

    fold_tprs.append(interpolated_tpr)
    fold_aucs.append(roc_display.roc_auc)

mean_tpr = np.mean(fold_tprs, axis=0)
mean_tpr[-1] = 1.0
mean_train_auc = auc(mean_fpr, mean_tpr)

plt.close("all")

In [ ]:
risk_model.fit(
    risk_train_expression,
    risk_train_labels,
    sample_weight=training_sample_weights
)

test_probabilities = risk_model.predict_proba(
    risk_test_expression
)[:, 1]

test_roc_display = RocCurveDisplay.from_predictions(
    risk_test_labels,
    test_probabilities
)

test_tpr = np.interp(
    mean_fpr,
    test_roc_display.fpr,
    test_roc_display.tpr
)
test_tpr[0] = 0.0
test_auc = test_roc_display.roc_auc

plt.close("all")

In [ ]:
plt.figure(figsize=(10, 9))
plt.rcParams["font.family"] = "Times"

plt.plot(
    mean_fpr,
    mean_tpr,
    color="red",
    label=f"Train (AUC={mean_train_auc:.2f})",
    linewidth=3,
    alpha=0.8
)

plt.plot(
    mean_fpr,
    test_tpr,
    color="blue",
    label=f"Test (AUC={test_auc:.2f})",
    linewidth=3,
    alpha=0.8
)

plt.plot([0, 1], [0, 1], "k--", linewidth=2.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontsize=24)
plt.ylabel("True Positive Rate", fontsize=24)
plt.legend(loc="lower right", prop={"size": 24})
plt.tick_params(axis="both", which="major", labelsize=20)
plt.savefig(
    figure_dir / "risk_model_ROC.pdf",
    bbox_inches="tight"
)
plt.show()

## Out-of-fold metrics and precision–recall curve

In [ ]:
risk_train_array = np.asarray(risk_train_labels).astype(int)
risk_train_weights = np.asarray([
    class_weight_by_label[label]
    for label in risk_train_array
])

oof_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_oof_probabilities = np.zeros(len(risk_train_array))


def get_rows(data, indices):
    if hasattr(data, "iloc"):
        return data.iloc[indices]
    return data[indices]


for train_index, validation_index in oof_cv.split(
    risk_train_expression,
    risk_train_array
):
    fold_model = clone(risk_model)

    fold_model.fit(
        get_rows(risk_train_expression, train_index),
        risk_train_array[train_index],
        sample_weight=risk_train_weights[train_index]
    )

    train_oof_probabilities[validation_index] = fold_model.predict_proba(
        get_rows(risk_train_expression, validation_index)
    )[:, 1]

print(
    "OOF train ROC-AUC:",
    roc_auc_score(risk_train_array, train_oof_probabilities)
)

print(
    "OOF train AP:",
    average_precision_score(
        risk_train_array,
        train_oof_probabilities
    )
)

In [ ]:
risk_model.fit(
    risk_train_expression,
    risk_train_array,
    sample_weight=risk_train_weights
)

test_probabilities = risk_model.predict_proba(
    risk_test_expression
)[:, 1]

print(
    "Test ROC-AUC:",
    roc_auc_score(risk_test_labels, test_probabilities)
)

print(
    "Test AP:",
    average_precision_score(risk_test_labels, test_probabilities)
)

In [ ]:
risk_test_array = np.asarray(risk_test_labels).ravel()

train_precision, train_recall, _ = precision_recall_curve(
    risk_train_array,
    train_oof_probabilities,
    pos_label=1
)

test_precision, test_recall, _ = precision_recall_curve(
    risk_test_array,
    test_probabilities,
    pos_label=1
)

train_auprc = auc(train_recall, train_precision)
test_auprc = auc(test_recall, test_precision)

train_baseline = (risk_train_array == 1).mean()
test_baseline = (risk_test_array == 1).mean()

In [ ]:
plt.figure(figsize=(10, 9))
plt.rcParams["font.family"] = "Times"

plt.step(
    train_recall,
    train_precision,
    where="post",
    linewidth=3,
    color="red",
    label=f"Train (AUPRC = {train_auprc:.2f})"
)

plt.step(
    test_recall,
    test_precision,
    where="post",
    linewidth=3,
    color="blue",
    label=f"Test (AUPRC = {test_auprc:.2f})"
)

plt.axhline(
    y=train_baseline,
    linestyle="--",
    linewidth=1.5,
    color="gray",
    label=f"Training baseline = {train_baseline:.2f}"
)

if abs(train_baseline - test_baseline) > 0.001:
    plt.axhline(
        y=test_baseline,
        linestyle=":",
        linewidth=1.5,
        color="gray",
        label=f"Test baseline = {test_baseline:.2f}"
    )

plt.xlabel("Recall", fontsize=24)
plt.ylabel("Precision", fontsize=24)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.grid(alpha=0.3)
plt.legend(loc="lower right", prop={"size": 20})
plt.tick_params(axis="both", which="major", labelsize=20)
plt.tight_layout()
plt.savefig(
    figure_dir / "risk_model_precision_recall.pdf",
    bbox_inches="tight"
)
plt.show()

## SHAP model interpretation

In [ ]:
check_is_fitted(risk_model)


def convert_to_dataframe(data, reference_columns=None):
    if isinstance(data, pd.DataFrame):
        return data.copy()

    if reference_columns is not None:
        columns = list(reference_columns)
    elif hasattr(risk_model, "feature_names_in_"):
        columns = list(risk_model.feature_names_in_)
    else:
        columns = [
            f"Feature_{i + 1}"
            for i in range(np.asarray(data).shape[1])
        ]

    return pd.DataFrame(
        np.asarray(data),
        columns=columns
    )


shap_train = convert_to_dataframe(risk_train_expression)
shap_test = convert_to_dataframe(
    risk_test_expression,
    reference_columns=shap_train.columns
)

In [ ]:
background_size = min(200, len(shap_train))

shap_background = shap.sample(
    shap_train,
    background_size,
    random_state=42
)

shap_explainer = shap.TreeExplainer(
    risk_model,
    data=shap_background,
    feature_perturbation="interventional",
    model_output="probability"
)

shap_values = shap_explainer(shap_test)

In [ ]:
def get_positive_class_explanation(explanation):
    if explanation.values.ndim == 2:
        return explanation

    if explanation.values.ndim == 3:
        return explanation[:, :, 1]

    raise ValueError(
        "Unexpected SHAP output shape: "
        f"{explanation.values.shape}"
    )


shap_positive = get_positive_class_explanation(shap_values)

print("SHAP values shape:", shap_positive.values.shape)
print("Test-data shape:", shap_test.shape)

In [ ]:
model_probabilities = risk_model.predict_proba(shap_test)[:, 1]

shap_reconstructed_probabilities = (
    np.asarray(shap_positive.base_values).ravel()
    + shap_positive.values.sum(axis=1)
)

maximum_difference = np.max(
    np.abs(
        model_probabilities
        - shap_reconstructed_probabilities
    )
)

print(
    "Maximum difference between model and SHAP probabilities:",
    maximum_difference
)

In [ ]:
plt.figure()

shap.plots.beeswarm(
    shap_positive,
    max_display=14,
    show=False
)

plt.xlabel(
    "SHAP value: effect on predicted high-risk probability",
    fontsize=12
)
plt.tight_layout()
plt.savefig(
    shap_output_dir / "risk_SHAP_beeswarm.pdf",
    bbox_inches="tight"
)
plt.show()

In [ ]:
plt.figure()

shap.plots.bar(
    shap_positive,
    max_display=14,
    show=False
)

plt.xlabel("Mean absolute SHAP value", fontsize=12)
plt.tight_layout()
plt.savefig(
    shap_output_dir / "risk_SHAP_global_importance.pdf",
    bbox_inches="tight"
)
plt.show()

In [ ]:
mean_absolute_shap = np.abs(shap_positive.values).mean(axis=0)
mean_signed_shap = shap_positive.values.mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": shap_test.columns,
    "Mean_absolute_SHAP": mean_absolute_shap,
    "Mean_signed_SHAP": mean_signed_shap
})

shap_importance = shap_importance.sort_values(
    "Mean_absolute_SHAP",
    ascending=False
).reset_index(drop=True)

shap_importance["Rank"] = np.arange(len(shap_importance)) + 1
shap_importance = shap_importance[
    [
        "Rank",
        "Feature",
        "Mean_absolute_SHAP",
        "Mean_signed_SHAP"
    ]
]

shap_importance.to_csv(
    shap_output_dir / "risk_SHAP_feature_importance.csv",
    index=False
)

shap_importance.round(5)

### Force plot for the reported hold-out sample

In [ ]:
def shap_block_force_plot(
    explanation,
    sample_name="Sample 1",
    panel_label=None,
    max_individual_features=6,
    positive_color="#F4CF3A",   # yellow
    negative_color="#A72A64",   # magenta
    output_path=None,
    probability_scale=True
):
    """
    Create a publication-style block SHAP force plot.

    Parameters
    ----------
    explanation : shap.Explanation
        One-dimensional SHAP explanation for one sample, for example:
        shap_positive[sample_index]

    sample_name : str
        Text displayed in the plot title.

    panel_label : str or None
        Optional panel letter, such as "E".

    max_individual_features : int
        Maximum number of individual features shown.
        Remaining features are grouped by contribution direction.

    positive_color : str
        Color for features increasing the class-1 prediction.

    negative_color : str
        Color for features decreasing the class-1 prediction.

    output_path : str, Path, or None
        PDF or PNG output filename.

    probability_scale : bool
        If True, the x-axis is labelled as predicted probability.
    """

    # --------------------------------------------------------
    # Extract SHAP information
    # --------------------------------------------------------
    shap_values = np.asarray(explanation.values).reshape(-1)
    base_value = float(
        np.asarray(explanation.base_values).reshape(-1)[0]
    )

    prediction = base_value + shap_values.sum()

    if explanation.feature_names is not None:
        feature_names = list(explanation.feature_names)
    else:
        feature_names = [
            f"Feature {i + 1}"
            for i in range(len(shap_values))
        ]

    if explanation.data is not None:
        feature_values = np.asarray(
            explanation.data
        ).reshape(-1)
    else:
        feature_values = np.repeat(
            np.nan,
            len(shap_values)
        )

    # --------------------------------------------------------
    # Retain major features and group smaller features
    # --------------------------------------------------------
    importance_order = np.argsort(
        np.abs(shap_values)
    )[::-1]

    keep_indices = importance_order[
        :max_individual_features
    ]

    other_indices = importance_order[
        max_individual_features:
    ]

    blocks = []

    for feature_index in keep_indices:

        blocks.append({
            "name": feature_names[feature_index],
            "feature_value": feature_values[feature_index],
            "shap_value": shap_values[feature_index],
            "is_group": False
        })

    # Group omitted positive contributions
    other_positive = [
        index for index in other_indices
        if shap_values[index] > 0
    ]

    if len(other_positive) > 0:

        blocks.append({
            "name": f"{len(other_positive)} other positive features",
            "feature_value": np.nan,
            "shap_value": shap_values[
                other_positive
            ].sum(),
            "is_group": True
        })

    # Group omitted negative contributions
    other_negative = [
        index for index in other_indices
        if shap_values[index] < 0
    ]

    if len(other_negative) > 0:

        blocks.append({
            "name": f"{len(other_negative)} other negative features",
            "feature_value": np.nan,
            "shap_value": shap_values[
                other_negative
            ].sum(),
            "is_group": True
        })

    positive_blocks = [
        block for block in blocks
        if block["shap_value"] > 0
    ]

    negative_blocks = [
        block for block in blocks
        if block["shap_value"] < 0
    ]

    # Small-to-large from left toward prediction
    positive_blocks.sort(
        key=lambda block: abs(block["shap_value"])
    )

    # Largest negative contribution next to prediction
    negative_blocks.sort(
        key=lambda block: abs(block["shap_value"]),
        reverse=True
    )

    total_positive = sum(
        block["shap_value"]
        for block in positive_blocks
    )

    total_negative_magnitude = sum(
        abs(block["shap_value"])
        for block in negative_blocks
    )

    left_limit_data = prediction - total_positive
    right_limit_data = prediction + total_negative_magnitude

    full_range = right_limit_data - left_limit_data

    if full_range == 0:
        full_range = 1.0

    padding = full_range * 0.10

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(12, 7)
    )

    plt.rcParams["font.family"] = "Times New Roman"

    bar_center = 1.0
    bar_height = 0.18

    lower_y = bar_center - bar_height / 2
    upper_y = bar_center + bar_height / 2

    def make_block(
        x_start,
        x_end,
        direction,
        color
    ):
        """
        Construct a chevron-like block.
        """

        width = abs(x_end - x_start)

        chevron_width = min(
            width * 0.15,
            full_range * 0.018
        )

        if direction == "right":

            vertices = [
                (x_start, lower_y),
                (x_end - chevron_width, lower_y),
                (x_end, bar_center),
                (x_end - chevron_width, upper_y),
                (x_start, upper_y),
                (x_start + chevron_width, bar_center)
            ]

        else:

            vertices = [
                (x_start, bar_center),
                (x_start + chevron_width, lower_y),
                (x_end, lower_y),
                (x_end - chevron_width, bar_center),
                (x_end, upper_y),
                (x_start + chevron_width, upper_y)
            ]

        polygon = Polygon(
            vertices,
            closed=True,
            facecolor=color,
            edgecolor="#333333",
            linewidth=1.5,
            zorder=3
        )

        ax.add_patch(polygon)

    def format_feature_value(value):

        if not np.isfinite(value):
            return ""

        if abs(value) >= 100:
            return f"{value:.1f}"

        if abs(value) >= 10:
            return f"{value:.2f}"

        return f"{value:.3f}".rstrip("0").rstrip(".")

    def add_feature_annotation(
        block,
        x_center,
        block_number,
        direction
    ):

        if block["is_group"]:

            label = block["name"]

        else:

            formatted_value = format_feature_value(
                block["feature_value"]
            )

            label = (
                f"{block['name']}={formatted_value}"
                if formatted_value
                else block["name"]
            )

        # Alternate heights to reduce overlap
        annotation_y = (
            1.28 if block_number % 2 == 0
            else 1.20
        )

        horizontal_shift = 0

        # Small outward shift for narrow blocks
        if direction == "right":
            horizontal_shift = -0.005 * full_range
        else:
            horizontal_shift = 0.005 * full_range

        ax.annotate(
            label,
            xy=(x_center, upper_y),
            xytext=(
                x_center + horizontal_shift,
                annotation_y
            ),
            ha="center",
            va="bottom",
            fontsize=10,
            color="#222222",
            arrowprops={
                "arrowstyle": "-",
                "color": "#B8B8B8",
                "linewidth": 1.0
            },
            zorder=5
        )

    # --------------------------------------------------------
    # Draw positive blocks
    # --------------------------------------------------------
    positive_cursor = (
        prediction - total_positive
    )

    block_number = 0

    for block in positive_blocks:

        x_start = positive_cursor
        x_end = x_start + block["shap_value"]

        make_block(
            x_start=x_start,
            x_end=x_end,
            direction="right",
            color=positive_color
        )

        x_center = (x_start + x_end) / 2

        ax.text(
            x_center,
            bar_center,
            f"+{block['shap_value']:.3f}",
            ha="center",
            va="center",
            fontsize=11,
            color="#222222",
            zorder=6
        )

        add_feature_annotation(
            block,
            x_center,
            block_number,
            direction="right"
        )

        positive_cursor = x_end
        block_number += 1

    # --------------------------------------------------------
    # Draw negative blocks
    # --------------------------------------------------------
    negative_cursor = prediction

    for block in negative_blocks:

        contribution_width = abs(
            block["shap_value"]
        )

        x_start = negative_cursor
        x_end = x_start + contribution_width

        make_block(
            x_start=x_start,
            x_end=x_end,
            direction="left",
            color=negative_color
        )

        x_center = (x_start + x_end) / 2

        ax.text(
            x_center,
            bar_center,
            f"{block['shap_value']:.3f}",
            ha="center",
            va="center",
            fontsize=11,
            color="white",
            zorder=6
        )

        add_feature_annotation(
            block,
            x_center,
            block_number,
            direction="left"
        )

        negative_cursor = x_end
        block_number += 1

    # --------------------------------------------------------
    # Baseline and final prediction
    # --------------------------------------------------------
    ax.vlines(
        base_value,
        ymin=0.63,
        ymax=0.94,
        color="#555555",
        linestyle=(0, (2, 2)),
        linewidth=1.5,
        zorder=2
    )

    ax.text(
        base_value,
        0.58,
        f"E[f(x)] = {base_value:.3f}",
        ha="center",
        va="top",
        fontsize=11
    )

    ax.vlines(
        prediction,
        ymin=0.78,
        ymax=0.94,
        color="#555555",
        linestyle=(0, (2, 2)),
        linewidth=1.5,
        zorder=2
    )

    prediction_label = (
        f"f(x) = {prediction:.3f}"
    )

    ax.text(
        prediction,
        0.73,
        prediction_label,
        ha="center",
        va="top",
        fontsize=11,
        fontweight="bold"
    )

    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------
    title = f"SHAP Force Plot – {sample_name}"

    if panel_label:
        title = f"{panel_label}. {title}"

    ax.set_title(
        title,
        loc="left",
        fontsize=15,
        fontweight="bold",
        pad=16
    )

    if probability_scale:
        ax.set_xlabel(
            "Model output: predicted high-risk probability",
            fontsize=13
        )
    else:
        ax.set_xlabel(
            "Model output",
            fontsize=13
        )

    ax.set_xlim(
        left_limit_data - padding,
        right_limit_data + padding
    )

    ax.set_ylim(
        0.48,
        1.43
    )

    ax.set_yticks([1])
    ax.set_yticklabels(["1"], fontsize=11)

    ax.tick_params(
        axis="x",
        labelsize=11
    )

    ax.grid(
        axis="x",
        alpha=0.20,
        linewidth=1
    )

    ax.grid(
        axis="y",
        alpha=0.15,
        linewidth=1
    )

    # Box around the figure
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color("#222222")

    plt.tight_layout()

    if output_path is not None:

        output_path = Path(output_path)

        fig.savefig(
            output_path,
            bbox_inches="tight",
            dpi=600
        )

    return fig, ax

In [ ]:
sample_index = 10

fig, ax = shap_block_force_plot(
    explanation=shap_positive[sample_index],
    sample_name=f"Test sample {sample_index}",
    max_individual_features=6,
    output_path=shap_output_dir / "risk_SHAP_force_sample_10.pdf"
)

plt.show()